In [8]:
import gurobipy as gp
from gurobipy import GRB

In [9]:
# 1) Datos de la instancia

N = 5
p_j = [14, 75, 23, 69, 19]       # duraciones
w_j = [199, 149, 40, 59, 78]     # beneficios/prioridades
h = 199                              # horizonte

# Parámetros de f(t) (función convexa lineal por tramos)
alpha = 60.0
beta = 8.5
v = 60.0

# Índices 1..N (más cómodo para leer resultados)
J = range(1, N + 1)
p = {j: p_j[j - 1] for j in J}
w = {j: w_j[j - 1] for j in J}

# Conjuntos de tiempos factibles de inicio T_j = {0,...,h-p_j}
T = {j: range(0, h - p[j] + 1) for j in J}

# Precedencias (i,j) en A. Para esta instancia pequeña: vacío.
A = [(1,3), (2,3)]   # ejemplo: A = [(1,3), (2,5)]

In [10]:
# ==========================================
# 2) f(t) y su primitiva para integrar exacto
# ==========================================
def f_piecewise(t: float) -> float:
    """f(t) de la ecuación (4.6)."""
    if 0 <= t <= v:
        return alpha - (alpha / v) * t
    elif v < t <= h:
        return (beta / v) * t - beta
    else:
        return 0.0

def F_primitive(t: float) -> float:
    """
    Primitiva de f(t), truncando fuera de [0,h]
    para poder calcular integrales exactas como F(b)-F(a).
    """
    if t <= 0:
        return 0.0
    if t <= v:
        # ∫ (alpha - alpha/v * s) ds = alpha*s - alpha/(2v)*s^2
        return alpha * t - (alpha / (2.0 * v)) * t * t
    else:
        # + ∫_v^t (beta/v * s - beta) ds +constante para continuidad
        return (beta / (2.0 * v)) * t * t - beta * t + (v*(alpha + beta))/2.0

def integral_f(a: float, b: float) -> float:
    """∫_a^b f(u)du exacta."""
    return F_primitive(b) - F_primitive(a)


In [11]:
# 3) Coeficientes del objetivo: q_{j,t} = w_j * ∫_t^{t+p_j} f
q = {(j, t): w[j] * integral_f(t, t + p[j]) for j in J for t in T[j]}

In [13]:
# =====================
# 4) Construcción MILP
# =====================
m = gp.Model("time_indexed_q_model")
m.Params.OutputFlag = 1  # cambia a 0 si no quieres logs

# Variables binarias x_{j,t} = 1 si actividad j inicia en t
x = m.addVars(((j, t) for j in J for t in T[j]), vtype=GRB.BINARY, name="x")

# (3.2) Maximizar sum_j sum_t q_{j,t} x_{j,t}
m.setObjective(gp.quicksum(q[j, t] * x[j, t] for j in J for t in T[j]), GRB.MAXIMIZE)

# (3.3) Presupuesto total de tiempo
m.addConstr(
    gp.quicksum(p[j] * x[j, t] for j in J for t in T[j]) <= h,
    name="time_budget"
)

# (3.4) Cada actividad a lo más una vez
for j in J:
    m.addConstr(
        gp.quicksum(x[j, t] for t in T[j]) <= 1,
        name=f"at_most_once[{j}]"
    )

# (3.5) No solapamiento
m.addConstrs(
    (
        gp.quicksum(
            x[j, tp]
            for j in J
            for tp in range(
                max(0, t - p[j] + 1),
                min(h - p[j] + 1, t + 1)
            )
        ) <= 1
        for t in range(0, h)
    ),
    name="capacity"
)


# (3.6) Precedencias Flexible
# x_{j,t} <= sum_{t'=0}^{t-p_i} x_{i,t'}   ∀(i,j)∈A, ∀t∈T_j ∩ [p_i, h-p_j]
#m.addConstrs(
#    (
#        x[j,t] <= gp.quicksum(x[i,tp] for tp in range(0, t-p[i] + 1 ))
#        for (i,j) in A
#        for t in range(p[i], h-p[j] + 1), 
#        name=f"precedence_flexible[{i},{j},{t}]"
#    )
#)

# (3.6) Preferencia Estricta
m.addConstrs(
    (
        gp.quicksum((t+p[i]) * x[i,t] for t in T[i]) <=
        gp.quicksum(tp * x[j,tp] for tp in T[j])
        for (i,j) in A
        
    ),
    name="precedence_strict"
)



m.update()

Set parameter OutputFlag to value 1


In [14]:
# 5) Resolver

m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 155H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 207 rows, 800 columns and 30791 nonzeros
Model fingerprint: 0x1f9e5225
Variable types: 0 continuous, 800 integer (800 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+03, 3e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+02]
Found heuristic solution: objective 97997.891667
Presolve removed 54 rows and 228 columns
Presolve time: 0.05s
Presolved: 153 rows, 572 columns, 20148 nonzeros
Variable types: 0 continuous, 572 integer (572 binary)

Root relaxation: objective 3.987459e+05, 88 iterations, 0.01 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

In [15]:
# =====================
# 6) Reporte de solución
# =====================
if m.Status == GRB.OPTIMAL:
    print("\n=== SOLUCIÓN ÓPTIMA ===")
    print(f"Valor objetivo = {m.ObjVal:.6f}")

    selected = []
    for j in J:
        for t in T[j]:
            if x[j, t].X > 0.5:
                selected.append((j, t, t + p[j], p[j], w[j], q[j, t]))

    selected.sort(key=lambda z: z[1])

    print("\nActividades seleccionadas (ordenadas por inicio):")
    for (j, start, end, dur, wj, qjt) in selected:
        print(f"  Act {j}: inicio={start:3d}, fin={end:3d}, p={dur:3d}, w={wj:3d}, q_jt={qjt:.6f}")

    selected_ids = {j for (j, *_rest) in selected}
    omitted = [j for j in J if j not in selected_ids]
    print(f"\nActividades NO seleccionadas: {omitted}")

    # Mostrar huecos (idle), por si te interesa
    print("\nHuecos (idle):")
    current = 0
    for (_, start, end, *_rest) in selected:
        if start > current:
            print(f"  [{current}, {start})  duración={start-current}")
        current = end
    if current < h:
        print(f"  [{current}, {h})  duración={h-current}")

elif m.Status == GRB.INFEASIBLE:
    print("El modelo es infactible.")
else:
    print(f"Estado del solver: {m.Status}")


=== SOLUCIÓN ÓPTIMA ===
Valor objetivo = 382068.316667

Actividades seleccionadas (ordenadas por inicio):
  Act 1: inicio=  0, fin= 14, p= 14, w=199, q_jt=147658.000000
  Act 2: inicio= 14, fin= 89, p= 75, w=149, q_jt=166518.054167
  Act 3: inicio=107, fin=130, p= 23, w= 40, q_jt=7624.500000
  Act 4: inicio=130, fin=199, p= 69, w= 59, q_jt=60267.762500

Actividades NO seleccionadas: [5]

Huecos (idle):
  [89, 107)  duración=18
